# Transformer English-to-Chinese Translation - Latest Colab

本 Notebook 會重新 clone 最新 Repo、生成大量訓練資料、重新訓練 Transformer，並使用 beam search 測試英翻中。

已修正：
- 不再使用 `!echo '\n...'`，避免 Colab 出現 unterminated string literal。
- 重新生成大型 dataset。
- 使用最新 config，避免 learning rate 過大造成只輸出句號。
- 推論時使用 beam search。

注意：本專案不使用 HuggingFace，也不使用任何 pretrained model。

## 1. 檢查 GPU

請先到 `Runtime → Change runtime type → T4 GPU` 開啟 GPU。

In [ ]:
import torch

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: GPU is not enabled. Training will be slower.')

## 2. Clone 最新 Repo

每次都會先刪掉舊資料夾，避免吃到 Colab cache 或舊模型。

In [ ]:
!rm -rf transformer-en-zh
!git clone https://github.com/slurpee0227/transformer-en-zh.git
%cd transformer-en-zh
!git log --oneline -5

## 3. 安裝套件

In [ ]:
!pip install -r requirements.txt

## 4. 重新生成大量訓練資料

此步驟會重新產生：

- `data/train.en`
- `data/train.zh`
- `data/valid.en`
- `data/valid.zh`

In [ ]:
!python src/generate_large_dataset.py

## 5. 檢查資料集

這裡使用 Python `print()`，不使用 `!echo '\n...'`，避免先前的 syntax error。

In [ ]:
print('===== train.en =====')
!head -5 data/train.en

print('')
print('===== train.zh =====')
!head -5 data/train.zh

print('')
print('===== line count =====')
!wc -l data/train.en data/train.zh data/valid.en data/valid.zh

## 6. 確認 Config

確認 learning rate 已經不是 1.0，避免模型崩成只輸出句號。

In [ ]:
!grep -n "LEARNING_RATE" src/config.py
!grep -n "EPOCHS" src/config.py
!grep -n "LABEL_SMOOTHING" src/config.py

## 7. 重新訓練 Transformer

請務必重新訓練，不要使用舊的 `.pt`，舊模型可能是用錯誤 learning rate 訓練出的。

In [ ]:
!rm -f models/transformer_zh_en.pt
!rm -rf checkpoints runs output/loss_curve.png
!python src/train.py

## 8. 確認輸出檔案

In [ ]:
print('===== models =====')
!ls -lh models

print('')
print('===== checkpoints =====')
!ls -lh checkpoints | head

print('')
print('===== output =====')
!ls -lh output

## 9. Beam Search 推論測試

In [ ]:
!python src/inference.py --sentence "I love artificial intelligence." --beam_search
!python src/inference.py --sentence "The model is training now." --beam_search
!python src/inference.py --sentence "The system stores the data." --beam_search
!python src/inference.py --sentence "The engineer checks the machine." --beam_search

## 10. 顯示 Loss Curve

In [ ]:
from IPython.display import Image, display

display(Image(filename='output/loss_curve.png'))

## 11. 啟動 TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 12. 下載繳交需要的檔案

請把下載的 `.pt` 放回作業 zip 的 `models/` 資料夾中。

In [ ]:
from google.colab import files

files.download('models/transformer_zh_en.pt')
files.download('output/loss_curve.png')